# Nebius Legal Fine-Tuning Notebook

This notebook is adapted for `legislation_qa_clean.jsonl` and uses a direct Nebius Token Factory fine-tuning workflow instead of the earlier customer-support and Data Lab pipeline.


## Prerequisites

- Set `NEBIUS_API_KEY` in the environment before running the notebook.
- Keep `legislation_qa_clean.jsonl` in the same directory as this notebook.
- The helper module `launch_legal_finetune.py` contains the validation, upload, and fine-tuning logic used below.


In [ ]:
# Uncomment if this kernel still needs dependencies.
# %pip install -q openai


In [ ]:
import json
import os
from pathlib import Path

from launch_legal_finetune import (
    DEFAULT_BASE_URL,
    DEFAULT_MODEL,
    DEFAULT_OUTPUT_PATH,
    STATE_PATH,
    build_client,
    create_finetune_job,
    load_state,
    sanitize_dataset,
    save_state,
    upload_training_file,
    wait_for_job,
)

DATASET_PATH = Path("legislation_qa_clean.jsonl")
CLEAN_DATASET_PATH = DEFAULT_OUTPUT_PATH
BASE_URL = DEFAULT_BASE_URL
MODEL = DEFAULT_MODEL
JOB_SUFFIX = "legislation-qa-lora"
HYPERPARAMETERS = {
    "n_epochs": 4,
    "learning_rate": 1e-5,
    "lora": True,
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "packing": True,
}

API_KEY = os.environ.get("NEBIUS_API_KEY")
if not API_KEY:
    raise RuntimeError("Set NEBIUS_API_KEY in your environment before running this notebook.")

client = build_client(api_key=API_KEY, base_url=BASE_URL)
print(json.dumps({
    "dataset": str(DATASET_PATH),
    "clean_dataset": str(CLEAN_DATASET_PATH),
    "state_path": str(STATE_PATH),
    "model": MODEL,
}, indent=2))


## Step 1 - Validate and sanitize the legal dataset

The source file is almost ready for Nebius as-is, but this step repairs malformed message roles and writes a clean training file for upload.


In [ ]:
report = sanitize_dataset(DATASET_PATH, CLEAN_DATASET_PATH)
state = load_state()
state.update({
    "dataset_path": str(DATASET_PATH),
    "clean_dataset_path": str(CLEAN_DATASET_PATH),
    "sanitization_report": report,
    "last_completed_step": 1,
})
save_state(state)
report


## Step 2 - Upload the cleaned training file to Nebius Token Factory


In [ ]:
training_file = upload_training_file(client, CLEAN_DATASET_PATH)
state = load_state()
state.update({
    "training_file_id": training_file.id,
    "last_completed_step": 2,
})
save_state(state)
{
    "training_file_id": training_file.id,
    "filename": training_file.filename,
    "bytes": getattr(training_file, "bytes", None),
}


## Step 3 - Launch the legal-domain LoRA fine-tuning job


In [ ]:
training_file_id = load_state()["training_file_id"]
job = create_finetune_job(
    client,
    training_file_id=training_file_id,
    model=MODEL,
    suffix=JOB_SUFFIX,
    hyperparameters=HYPERPARAMETERS,
    seed=42,
)
state = load_state()
state.update({
    "fine_tuning_job_id": job.id,
    "fine_tuning_status": job.status,
    "last_completed_step": 3,
})
save_state(state)
{
    "job_id": job.id,
    "status": job.status,
    "model": job.model,
    "training_file": job.training_file,
}


## Step 4 - Monitor the fine-tuning job

Run the next cell to poll Nebius until the job reaches a terminal state.


In [ ]:
job_id = load_state()["fine_tuning_job_id"]
final_job = wait_for_job(client, job_id, poll_seconds=30)
state = load_state()
state.update({
    "fine_tuning_status": final_job.status,
    "last_completed_step": 4,
})
save_state(state)
{
    "job_id": final_job.id,
    "status": final_job.status,
    "trained_tokens": getattr(final_job, "trained_tokens", None),
    "fine_tuned_model": getattr(final_job, "fine_tuned_model", None),
}


## Notes

- The sanitization step repairs malformed roles like the first record in `legislation_qa_clean.jsonl` where the user prompt was stored in the `role` field.
- State is persisted to `artifacts/legal_finetune_state.json`, so reruns can reuse file and job IDs.
- If the job succeeds and you want deployment next, use the `nebius-deploy-lora` workflow against the produced checkpoint.
